In [1]:
import pandas as pd
from bs4 import BeautifulSoup

In [2]:
ruta_archivo_noticias = "./../data_processed/conjunto_noticias_2.json"
df = pd.read_json(ruta_archivo_noticias)
df

,Link,Periódico,Fecha,Título,Subtítulo,Categoría,Contenido,cb,cb_score,cb_label
0,https://www.20minutos.es/internacional/estrech...,20minutos,2026-04-29,El estrecho Bab al Mandeb: todo sobre el otro ...,Por este estrecho pasa el 12% del comercio mar...,Internacional,La guerra en Irán ni acaba ni continúa. El con...,False,0.9273,NO Clickbait
1,https://www.20minutos.es/internacional/menos-c...,20minutos,2026-04-28,Tiroteo en el centro de Atenas: un hombre de 8...,La Policía griega ha arrestado al presunto aut...,Internacional,Un hombre hirió este martes a cinco personas e...,False,1.0000,NO Clickbait
2,https://www.20minutos.es/internacional/brutal-...,20minutos,2026-04-28,La brutal respuesta del narco a la detención d...,La captura Audias Flores Silva uno de los líde...,Internacional,El Gobierno del estado mexicano de Nayarit (oe...,False,0.5069,NO Clickbait
3,https://www.20minutos.es/internacional/carlos-...,20minutos,2026-04-29,Carlos III en la Corte de Trump,Starmer trata de usar la admiración y debilida...,Internacional,Cuatro días estará el Rey de Inglaterra en los...,False,0.5486,NO Clickbait
4,https://www.20minutos.es/internacional/senado-...,20minutos,2026-04-29,El Senado de EEUU falla en su intento de limit...,Con una votación de 47 votos a favor y 51 en c...,Internacional,Los demócratas del Senado de Estados Unidos ha...,False,1.0000,NO Clickbait
...,...,...,...,...,...,...,...,...,...,...
535,https://www.mediterraneodigital.com/espana/and...,Mediterráneo Digital,12-05-2026,Detenido un concejal del PSOE en Chiclana por ...,NaN,Nacional,Un concejal del PSOE de Chiclana y octavo teni...,False,1.0000,NO Clickbait
536,https://www.mediterraneodigital.com/sucesos-es...,Mediterráneo Digital,12-05-2026,A juicio a dos familias por casar a una niña p...,Indicios de trata de seres humanos y mendicidad,Nacional,El juez ve indicios de trata de seres humanos ...,False,1.0000,NO Clickbait
537,https://www.mediterraneodigital.com/sucesos-es...,Mediterráneo Digital,12-05-2026,A juicio tres inmigrantes ecuatorianos por la ...,NaN,Nacional,Tres hombres de nacionalidad ecuatoriana se se...,False,1.0000,NO Clickbait
538,https://www.mediterraneodigital.com/sucesos-es...,Mediterráneo Digital,12-05-2026,Un hombre discute con su amiga en Mallorca y l...,NaN,Nacional,La Audiencia Provincial de las Islas Baleares ...,True,0.9999,Clickbait


In [3]:
df["Contenido"]

0      La guerra en Irán ni acaba ni continúa. El con...
1      Un hombre hirió este martes a cinco personas e...
2      El Gobierno del estado mexicano de Nayarit (oe...
3      Cuatro días estará el Rey de Inglaterra en los...
4      Los demócratas del Senado de Estados Unidos ha...
                             ...                        
535    Un concejal del PSOE de Chiclana y octavo teni...
536    El juez ve indicios de trata de seres humanos ...
537    Tres hombres de nacionalidad ecuatoriana se se...
538    La Audiencia Provincial de las Islas Baleares ...
539    Diez jóvenes de 15 y 16 años denuncian tocamie...
Name: Contenido, Length: 540, dtype: str

In [4]:
import re
import regex

## CARACTERES QUE HAY QUE ELIMINAR:
---

Tras una vista previa del texto guardado en el archivo JSON de `conjunto_noticias.json`, estos son los elementos del texto que se han detectado que deben o deberían eliminarse:
- **Comillas:** hay muchas formas diferentes que cada periódico utiliza para poner comillas, estas son: /""/, ««, '', "" y algunas más.  
- **Signos de interrogación y exclamación:** ¡! ¿?
- **Guiones:** --
- **URLs:**
- **Direcciones de correo y cuentas de twitter:** empiezan por @
- **Saltos de línea:** \n
- **Signos de puntuación:** , ; : .
- **Corchetes y paréntesis:** [] ()
- **Emojis**

In [5]:
# FUNCIONES PARA LIMPIAR TEXTO:

# Quitar URLs
def remove_urls(texto):
    patron = re.compile(r"https?://\S+")
    texto = patron.sub("", texto)
    patron = re.compile(r"\b[A-Za-z0-9.-]+\.(com|es|net|org)(/[A-Za-z0-9._\-]+)*")
    return patron.sub("", texto)

# Quitar emojis
def remove_emoji(text):
    emoji_pattern = re.compile("["
                           u"\U0001F600-\U0001F64F"  # emoticons
                           u"\U0001F300-\U0001F5FF"  # symbols & pictographs
                           u"\U0001F680-\U0001F6FF"  # transport & map symbols
                           u"\U0001F1E0-\U0001F1FF"  # flags 
                           u"\U00002702-\U000027B0"
                           u"\U000024C2-\U0001F251"
                           u"\U0001F900-\U0001F9FF"  # Emojis suplementarios
                           u"\U0001FA70-\U0001FAFF"  # Emojis más nuevos
                           "]+", flags=re.UNICODE)
    return emoji_pattern.sub(r'', text)

# Quitar comillas
# Como hay tantos tipos de comillas y puede darse el caso de que haya incluso más de las que se han visto, se usa
# {Quotation_Mark} que es una lista con todos los códigos de todas las comillas posibles dentro del estándar UNICODE
# OTRA OPCIÓN: usar patron = r'[\"\'«»“”‘’]', pero si hay algún tipo de comilla que no está en la lista, no lo quita
def remove_comillas(texto):
    patron = r'\p{Quotation_Mark}'
    return regex.sub(patron, "", texto)

# Quitar saltos de línea
def remove_saltos_linea(texto):
    return texto.replace("\n", "").replace("\r", "")

# Signos de puntuación
# Hay un argumento opcional para quitar o no los puntos, para tener la opción de dividir el texto en frases
def remove_puntuacion_basica(texto, quitar_puntos):
    patron = r"[,;:]"
    if quitar_puntos:
        patron = r"[.,;:]"
    return re.sub(patron, "", texto)

# Menciones tipo Twitter (por ejemplo @usuario_123)
def remove_menciones(texto):
    patron = r"@[A-Za-z0-9_]+"
    return re.sub(patron, "", texto)

# Caracteres especiales y raros
def remove_esp(texto):
    patron = r"[#@]"
    return re.sub(patron, "", texto)

# Signos de exclamación e interrogación
def remove_int_excl(texto):
    texto = re.sub(r'[¡!¿?]+', '.', texto)
    texto = re.sub(r'\.+', '.', texto)
    return texto

# Quitar frágmetos y etiquetas de código html, pues algunos RSS devuelven la noticia con las etiquetas
def remove_html(texto):
    return BeautifulSoup(texto, "html.parser").get_text(" ", strip=True)

# Paréntesis, corchetes y guiones
# Para los guiones, mejor sustituirlos por espacio, pues pueden aparecer en palabras compuestas (teórico-práctico) o para
# delimitar citas o información (-Una cita-). En el primer caso, elimnar el guion y no poner nada hace que se forme una
# palabra mal escrita, así que mejor poner espacio
def remove_delimitadores(texto):
    texto = re.sub(r'[\(\)\[\]]', '', texto)
    # Resulta que también hay varios tipos de guion diferentes, todos ellos se pueden quitar con la expresión regular
    # \p{Pd}
    return regex.sub(r"\p{Pd}", " ", texto)

# Después de haber eliminado muchos elementos del texto es normal que los espacios entre palabras ya no sean los correctos
# y que muchas palabras estén separadas por dos o más espacios, en vez de solo uno, así que debemos asegurar que 
# los espacios quedan bien
def clean_espacios(texto):
    # Reemplaza múltiples espacios por uno solo
    texto = re.sub(r"\s+", " ", texto)
    # Elimina espacios al inicio y al final
    return texto.strip()

In [6]:
# Vamos a hacer una función que ensamble todas las funciones de limpieza anteriores 
def Limpieza_texto(texto, minusculas=False, sign_excl_int=False, quitar_puntos=False):
    
    # Como algunas noticias no tienen subtítulo es posible que texto está vacío o sea none o NaN, así que hay que poner
    # un condicional
    if texto is None or pd.isna(texto):
        return ""

    # Puede ser que no queramos que el texto esté en minúsculas
    if minusculas:
        texto = texto.lower()
    
    # Puede ser que no queramos quitar los símbolos de exclamación e interrogación
    if sign_excl_int:
        texto = remove_int_excl(texto)

    texto = remove_urls(texto)
    texto = remove_menciones(texto)
    texto = remove_emoji(texto)
    texto = remove_comillas(texto)
    texto = remove_html(texto)
    texto = remove_esp(texto)
    texto = remove_saltos_linea(texto)
    texto = remove_puntuacion_basica(texto, quitar_puntos=quitar_puntos)
    texto = remove_delimitadores(texto)

    return clean_espacios(texto)

In [7]:
import os
import json

# - df: dataframe con el texto que vamos limpiar
# - ruta_origen: ruta local al archivo json donde se encuentra el dataframe con el texto que hay que limpiar
# - nombre_archivo: ruta donde queremos guardar el dataframe con el texto ya limpio
# - sustituir: en caso de que pueda existir un archivo con el mismo nombre para no correr el riesgo de sobreescribir y
# perder información importante, le ponemos que por defecto busque un nombre diferente

def Limpieza_y_guardado(ruta_origen, nombre_archivo, sustituir=False, 
                        minusculas=False, sign_excl_int=False, quitar_puntos=False):

    with open(ruta_origen, "r", encoding="utf-8") as f:
        df = pd.DataFrame(json.load(f))

    df['Contenido'] = df['Contenido'].apply(lambda x: Limpieza_texto(x, minusculas=minusculas, sign_excl_int=sign_excl_int, quitar_puntos=quitar_puntos))
    df['Título'] = df['Título'].apply(lambda x: Limpieza_texto(x, minusculas=minusculas, sign_excl_int=sign_excl_int, quitar_puntos=quitar_puntos))
    df['Subtítulo'] = df['Subtítulo'].apply(lambda x: Limpieza_texto(x, minusculas=minusculas, sign_excl_int=sign_excl_int, quitar_puntos=quitar_puntos))

    # Las dos siguientes líneas ensucian bastante la función solo es para que el nombre de un periódico aparezca 
    # con un formato correcto y sea legible
    df["Periódico"] = (df["Periódico"].str.replace(r"www\.([a-zA-Z0-9_-]+)\.(es|com)", 
                                               lambda m: m.group(1).capitalize(), regex=True))
    df["Periódico"] = df["Periódico"].str.replace(
    r"(.*?)(digital)",
    lambda m: f"{m.group(1)} {m.group(2).capitalize()}",
    regex=True
)

    nombre_final = nombre_archivo
    # Si no se debe sustituir, buscamos un nombre alternativo
    if not sustituir:
        contador = 1
        base, ext = os.path.splitext(nombre_archivo)

        while os.path.exists(nombre_final):
            nombre_final = f"{base}_{contador}{ext}"
            contador += 1

    # Guardar JSON
    data = df.to_dict(orient="records")
    texto = json.dumps(data, ensure_ascii=False, indent=4)

    with open(nombre_final, "w", encoding="utf-8") as f:
        f.write(texto)

    return nombre_final


In [8]:
# Primera prueba

ruta_archivo_noticias = "./../data_processed/conjunto_noticias_2.json"
ruta_guardado = "./../data_processed/conjunto_noticias_2_limpio.json"

Limpieza_y_guardado(ruta_origen=ruta_archivo_noticias,
                    nombre_archivo=ruta_guardado,
                    minusculas=True)

'./../data_processed/conjunto_noticias_2_limpio_1.json'

## Limpio + Sin Stopwords

---

In [9]:
#Cargamos librerías
import spacy
import nltk
from nltk.corpus import stopwords
import json
import os
import pandas as pd
import regex
import re

c:\Users\Usuario\anaconda3\envs\PLN\Lib\site-packages\requests\__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


In [10]:
# Cargamos herramientas de procesamiento

nltk.download('stopwords')               # Descargamos stopwords de NLTK
nlp = spacy.load("es_core_news_sm")      # Cargamos modelo spaCy para español para tokenizar
stop = set(stopwords.words('spanish'))   # Guardamos un set (más rápido de buscar que una lista) de stopwords en español en esa variable

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Usuario\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Para clickbait, algunas stopwords pueden ser informativas:

- por qué
- cómo
- qué
- este
- esto
- así
- no
- nunca
- nadie

Ejemplos:

- No creerás lo que pasó después
- Así usa tus impuestos el PSOE
- Este es el truco que nadie te cuenta

In [11]:
# Creamos una versión con algunas stop words conservadas
# Sobretodo, interrogativas, negaciones, demostrativos
stop = set(stopwords.words('spanish'))

conservar = {

    # Negaciones
    "no", "nunca", "jamás", "nadie", "nada", "ningún", "ninguna",

    # Interrogativos / exclamativos
    "qué", "que",
    "cómo", "como",
    "cuál", "cuáles",
    "quién", "quiénes",
    "cuándo", "cuando",
    "dónde", "donde",

    # Demostrativos
    "este", "esta", "estos", "estas",
    "ese", "esa", "esos", "esas",
    "aquel", "aquella",
    "esto", "eso", "aquello",

    # Pronombres apelativos
    "tú", "tu", "usted", "ustedes",
    "te", "ti",

    # Intensificadores / énfasis
    "muy", "más", "mas",
    "tan", "tanto",
    "solo", "sólo",
    "incluso",

    # Conectores frecuentes en clickbait
    "porque", "por",
    "si",
    "aunque",

    # Auxiliares y verbos muy típicos
    "puede", "puedes",
    "debe", "debes",
    "vas", "van",
    "tiene", "tienen",

    # Expresiones típicas de titulares
    "así",
    "esto",
    "todo"
}

stop_filtradas = stop - conservar

In [12]:
#probamos stop_filtradas
doc=nlp("No creerás lo que pasó después")
print([token.text for token in doc if token.text.lower() not in stop and token.is_alpha])
print([token.text for token in doc if token.text.lower() not in stop_filtradas and token.is_alpha])
doc=nlp("Así usan tus impuestos el PSOE")
print([token.text for token in doc if token.text.lower() not in stop and token.is_alpha])
print([token.text for token in doc if token.text.lower() not in stop_filtradas and token.is_alpha])
doc=nlp("Este es el truco que nadie te cuenta")
print([token.text for token in doc if token.text.lower() not in stop and token.is_alpha])
print([token.text for token in doc if token.text.lower() not in stop_filtradas and token.is_alpha])

['creerás', 'pasó', 'después']
['No', 'creerás', 'que', 'pasó', 'después']
['Así', 'usan', 'impuestos', 'PSOE']
['Así', 'usan', 'impuestos', 'PSOE']
['truco', 'nadie', 'cuenta']
['Este', 'truco', 'que', 'nadie', 'te', 'cuenta']


In [13]:
# Decidimos si queremos usar el filtro o no
conservar = False 

# Cargamos los datos
with open(ruta_guardado, "r", encoding="utf-8") as f:
    noticias = json.load(f)

In [14]:
# Procesar cada noticia

for noticia in noticias:
    for campo in ["Título", "Subtítulo", "Contenido"]:
        if campo in noticia:
            doc = nlp(noticia[campo]) #tokenizamos el contenido de cada campo dentro de cada noticia
            # Filtrar stopwords
            if conservar:
                sinstopwords = [token.text for token in doc if token.text.lower() not in stop and token.is_alpha]
            else:
                sinstopwords = [token.text for token in doc if token.text.lower() not in stop_filtradas and token.is_alpha]

            noticia[campo] = " ".join(sinstopwords)

In [15]:
# Previsualizamos
noticias_df = pd.DataFrame(noticias)
noticias_df.loc[:,["Título", "Subtítulo", "Contenido"]].head()

,Título,Subtítulo,Contenido
0,estrecho bab mandeb todo punto clave comercio ...,por este estrecho pasa comercio marítimo conte...,guerra irán acaba continúa conflicto unidos is...
1,tiroteo centro atenas hombre años deja cinco h...,policía griega arrestado presunto autor ataque...,hombre hirió este martes cinco personas atenas...
2,brutal respuesta narco detención capo jardiner...,captura audias flores silva líderes cjng zona ...,gobierno mexicano nayarit oeste país municipio...
3,carlos iii corte trump,starmer trata usar admiración debilidad trump ...,cuatro días rey inglaterra unidos como invitad...
4,senado eeuu falla intento limitar posibles acc...,votación votos favor iniciativa controlar acci...,demócratas senado unidos fracasado este martes...


In [16]:
# Comparamos con el original, necesario cargar df (Sección de Juan, segundo chunk)
df.loc[:,["Título", "Subtítulo", "Contenido"]].head()

,Título,Subtítulo,Contenido
0,El estrecho Bab al Mandeb: todo sobre el otro ...,Por este estrecho pasa el 12% del comercio mar...,La guerra en Irán ni acaba ni continúa. El con...
1,Tiroteo en el centro de Atenas: un hombre de 8...,La Policía griega ha arrestado al presunto aut...,Un hombre hirió este martes a cinco personas e...
2,La brutal respuesta del narco a la detención d...,La captura Audias Flores Silva uno de los líde...,El Gobierno del estado mexicano de Nayarit (oe...
3,Carlos III en la Corte de Trump,Starmer trata de usar la admiración y debilida...,Cuatro días estará el Rey de Inglaterra en los...
4,El Senado de EEUU falla en su intento de limit...,Con una votación de 47 votos a favor y 51 en c...,Los demócratas del Senado de Estados Unidos ha...


In [17]:
# Guardar resultado
if conservar:
    with open("./../data_processed/conjunto_noticias_procesado_1_2.json", "w", encoding="utf-8") as f:
        json.dump(noticias, f, ensure_ascii=False, indent=2)
else:
    with open("./../data_processed/conjunto_noticias_procesado_1_2_filter.json", "w", encoding="utf-8") as f:
        json.dump(noticias, f, ensure_ascii=False, indent=2)

## Limpio + Lematizado + Sin Stopwords

---

In [18]:
import spacy
import nltk
from nltk.corpus import stopwords
import json
nltk.download('stopwords')
nlp = spacy.load("es_core_news_sm")
stop = stopwords.words('spanish')

with open("./../data_processed/conjunto_noticias_limpio.json", "r", encoding="utf-8") as f:
    noticias = json.load(f)

# Procesar cada noticia
for noticia in noticias:
    for campo in ["Título", "Subtítulo", "Contenido"]:
        if campo in noticia:
            doc = nlp(noticia[campo])
            # Lematizar y filtrar stopwords
            lemmas = [token.lemma_ for token in doc if token.text.lower() not in stop and token.is_alpha]
            noticia[campo] = " ".join(lemmas)

# Guardar resultado
with open("./../data_processed/conjunto_noticias_procesado_1_3.json", "w", encoding="utf-8") as f:
    json.dump(noticias, f, ensure_ascii=False, indent=2)

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Usuario\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


## Limpio + Lematizado + Con Stopwords

---

In [19]:
import spacy
import nltk
from nltk.corpus import stopwords
import json
nltk.download('stopwords')
nlp = spacy.load("es_core_news_sm")
stop = stopwords.words('spanish')

with open("./../data_processed/conjunto_noticias_limpio.json", "r", encoding="utf-8") as f:
    noticias = json.load(f)

# Procesar cada noticia
for noticia in noticias:
    for campo in ["Título", "Subtítulo", "Contenido"]:
        if campo in noticia:
            doc = nlp(noticia[campo])
            # Lematizar y filtrar stopwords
            lemmas = [token.lemma_ for token in doc if token.is_alpha]
            noticia[campo] = " ".join(lemmas)

# Guardar resultado
with open("./../data_processed/conjunto_noticias_procesado_1_4.json", "w", encoding="utf-8") as f:
    json.dump(noticias, f, ensure_ascii=False, indent=2)

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Usuario\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


### Limpieza de los datos del agente

----

In [21]:
import os
import json
import re
import pandas as pd
import regex
from bs4 import BeautifulSoup

# ============================================================
# FUNCIONES PARA LIMPIAR TEXTO
# ============================================================

def remove_urls(texto):
    """Quitar URLs"""
    patron = re.compile(r"https?://\S+")
    texto = patron.sub("", texto)
    patron = re.compile(r"\b[A-Za-z0-9.-]+\.(com|es|net|org)(/[A-Za-z0-9._\-]+)*")
    return patron.sub("", texto)

def remove_emoji(text):
    """Quitar emojis"""
    emoji_pattern = re.compile("["
                           u"\U0001F600-\U0001F64F"
                           u"\U0001F300-\U0001F5FF"
                           u"\U0001F680-\U0001F6FF"
                           u"\U0001F1E0-\U0001F1FF"
                           u"\U00002702-\U000027B0"
                           u"\U000024C2-\U0001F251"
                           u"\U0001F900-\U0001F9FF"
                           u"\U0001FA70-\U0001FAFF"
                           "]+", flags=re.UNICODE)
    return emoji_pattern.sub(r'', text)

def remove_comillas(texto):
    """Quitar comillas (todos los tipos UNICODE)"""
    patron = r'\p{Quotation_Mark}'
    return regex.sub(patron, "", texto)

def remove_saltos_linea(texto):
    """Quitar saltos de línea"""
    return texto.replace("\n", "").replace("\r", "")

def remove_puntuacion_basica(texto, quitar_puntos=False):
    """Quitar signos de puntuación básica"""
    patron = r"[,;:]"
    if quitar_puntos:
        patron = r"[.,;:]"
    return re.sub(patron, "", texto)

def remove_menciones(texto):
    """Quitar menciones tipo Twitter"""
    patron = r"@[A-Za-z0-9_]+"
    return re.sub(patron, "", texto)

def remove_esp(texto):
    """Quitar caracteres especiales # @"""
    patron = r"[#@]"
    return re.sub(patron, "", texto)

def remove_int_excl(texto):
    """Quitar signos de exclamación e interrogación"""
    texto = re.sub(r'[¡!¿?]+', '.', texto)
    texto = re.sub(r'\.+', '.', texto)
    return texto

def remove_html(texto):
    """Quitar fragmentos y etiquetas HTML"""
    return BeautifulSoup(texto, "html.parser").get_text(" ", strip=True)

def remove_delimitadores(texto):
    """Quitar paréntesis, corchetes y guiones"""
    texto = re.sub(r'[\(\)\[\]]', '', texto)
    return regex.sub(r"\p{Pd}", " ", texto)

def clean_espacios(texto):
    """Normalizar espacios"""
    texto = re.sub(r"\s+", " ", texto)
    return texto.strip()

def Limpieza_texto(texto, minusculas=False, sign_excl_int=False, quitar_puntos=False):
    """Ensamble de todas las funciones de limpieza"""
    
    if texto is None or pd.isna(texto):
        return ""

    if minusculas:
        texto = texto.lower()
    
    if sign_excl_int:
        texto = remove_int_excl(texto)

    texto = remove_urls(texto)
    texto = remove_menciones(texto)
    texto = remove_emoji(texto)
    texto = remove_comillas(texto)
    texto = remove_html(texto)
    texto = remove_esp(texto)
    texto = remove_saltos_linea(texto)
    texto = remove_puntuacion_basica(texto, quitar_puntos=quitar_puntos)
    texto = remove_delimitadores(texto)

    return clean_espacios(texto)

# ============================================================
# JUNTAR Y LIMPIAR DATOS DEL AGENTE
# ============================================================

AGENTE_DIR = "./../agente/data/"
SALIDA_LIMPIA = "./../data_processed/datos_agente_limpio.json"

ARCHIVOS = [
    "20minutos.json",
    "ABC.json",
    "eldiario.json",
    "elhuffpost.json",
    "lavanguardia.json",
    "mediterraneodigital.json",
    "okdiario.json",
    "RTVE.json",
]

CAMPOS = ["Link", "Periódico", "Fecha", "Título", "Subtítulo", "Categoría", "Contenido", "cb", "cb_score", "cb_label"]

noticias_agente = []

print("Cargando datos del agente...")
for archivo in ARCHIVOS:
    ruta = os.path.join(AGENTE_DIR, archivo)
    with open(ruta, "r", encoding="utf-8") as f:
        datos = json.load(f)

    if isinstance(datos, dict):
        datos = list(datos.values())[0] if datos else []

    for noticia in datos:
        noticias_agente.append({campo: noticia.get(campo) for campo in CAMPOS})

    print(f"  {archivo:<30} {len(datos):>4} noticias")

print(f"\nTotal cargado: {len(noticias_agente)} noticias")

# Convertir a DataFrame para limpiar
df = pd.DataFrame(noticias_agente)

print("\nLimpiando textos...")
df['Contenido'] = df['Contenido'].apply(lambda x: Limpieza_texto(x, minusculas=True, sign_excl_int=True, quitar_puntos=False))
df['Título'] = df['Título'].apply(lambda x: Limpieza_texto(x, minusculas=True, sign_excl_int=True, quitar_puntos=False))
df['Subtítulo'] = df['Subtítulo'].apply(lambda x: Limpieza_texto(x, minusculas=True, sign_excl_int=True, quitar_puntos=False))

# Normalizar nombres de periódicos
df["Periódico"] = (df["Periódico"].str.replace(r"www\.([a-zA-Z0-9_-]+)\.(es|com)", 
                                               lambda m: m.group(1).capitalize(), regex=True))
df["Periódico"] = df["Periódico"].str.replace(
    r"(.*?)(digital)",
    lambda m: f"{m.group(1)} {m.group(2).capitalize()}",
    regex=True
)

# Guardar JSON limpio
os.makedirs("data_processed", exist_ok=True)
data = df.to_dict(orient="records")
with open(SALIDA_LIMPIA, "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=2)

print(f"\nArchivo limpio guardado en: {SALIDA_LIMPIA}")
print(f"Total: {len(df)} noticias")
print(f"Clickbait (GPT):    {sum(df['cb'])}")
print(f"No Clickbait (GPT): {len(df) - sum(df['cb'])}")
print(f"Periódicos: {sorted(df['Periódico'].unique().tolist())}")

Cargando datos del agente...
  20minutos.json                  141 noticias
  ABC.json                        141 noticias
  eldiario.json                   148 noticias
  elhuffpost.json                 205 noticias
  lavanguardia.json               143 noticias
  mediterraneodigital.json         94 noticias
  okdiario.json                   120 noticias
  RTVE.json                        78 noticias

Total cargado: 1070 noticias

Limpiando textos...

Archivo limpio guardado en: ./../data_processed/datos_agente_limpio.json
Total: 1070 noticias
Clickbait (GPT):    260
No Clickbait (GPT): 810
Periódicos: ['20minutos', 'ABC', 'ElDiario', 'HuffPost', 'La Vanguardia', 'Mediterráneo Digital', 'OkDiario', 'RTVE']
